<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/15-latent-variable-flow-energy-models.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **潜变量、流与能量模型** {#latent-variable-flow-energy-models}

第 14 章的自回归构造通过选择顺序使似然变得可处理。本章研究对同一个密度建模问题的三种不同回答。**潜变量模型**借助隐藏变量解释观测；**归一化流**通过可逆映射变换一个简单密度；**能量模型**为合理配置赋予较低的标量能量，而不要求归一化常数能够立即计算。

![潜变量、流与能量模型对密度和推断作出了不同承诺。](assets/dl15-model-family-map.svg){fig-align="center" width="78%" fig-alt="三个面板从数据分布表示方式比较潜变量模型、归一化流与能量模型。"}

*基于 [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114)、[RealNVP](https://arxiv.org/abs/1605.08803) 与 [Energy-Based Learning 教程](https://yann.lecun.org/exdb/publis/pdf/lecun-06.pdf) 绘制的原创综合图。*

全部实验使用 scikit-learn 收录的 [UCI Optical Recognition of Handwritten Digits 数据集](https://archive.ics.uci.edu/dataset/80/optical%2Brecognition%2Bof%2B)，DOI 为 [10.24432/C50P49](https://doi.org/10.24432/C50P49)，采用 CC BY 4.0 许可。固定的 70/15/15 划分只创建一次。自编码器与 VAE 处理相同的 64 维图像；后续的流、能量和 score 模型处理经过标准化的二维自编码器表示，使密度几何能够直接观察。这个低维阶段只演示机制，不是现代图像生成 benchmark。

<details>
<summary><strong>PyTorch：建立共享的图像与潜空间密度实验</strong></summary>

```python
import copy
import math
import os
import random

import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")


def seed_everything(seed=1515):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_images = torch.tensor(digits.images, dtype=torch.float32).flatten(1) / 16.0
all_labels = torch.tensor(digits.target, dtype=torch.long)
all_indices = np.arange(len(all_images))
train_idx, holdout_idx = train_test_split(
    all_indices, test_size=0.30, random_state=1515, stratify=digits.target
)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.50, random_state=1515,
    stratify=digits.target[holdout_idx],
)
train_x, train_y = all_images[train_idx], all_labels[train_idx]
val_x, val_y = all_images[val_idx], all_labels[val_idx]
test_x, test_y = all_images[test_idx], all_labels[test_idx]


def make_loader(images, batch_size=160, seed=1515):
    return DataLoader(
        TensorDataset(images), batch_size=batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )


assert all_images.shape == (1797, 64)
assert len(set(train_idx) & set(test_idx)) == 0
assert train_x.min() >= 0 and train_x.max() <= 1
print({"split": (len(train_x), len(val_x), len(test_x)),
       "input shape": tuple(train_x.shape), "classes": int(all_labels.unique().numel())})
```

</details>

表示模型训练不使用任何标签；标签仅保留给后续诊断探针。所有需要学习的归一化统计量只在训练表示上拟合，再原样用于验证、测试和生成点。


### **潜变量与隐藏结构** {#latent-variables-hidden-structure}

潜变量模型引入一个未观测变量 $z$：

$$
p_{\theta}(x)=\int p_{\theta}(x\mid z)p(z)\,dz.
$$

$z$ 可以表示类别、姿态、风格、主题、说话人身份、物理状态或其他隐藏原因。但这一分解不保证每个坐标都会获得人类可解释的含义：许多潜表示会产生相同的边缘分布 $p(x)$，旋转或置换也可能保持似然不变。因此，可解释性需要额外假设、监督、干预、架构偏置，或基于已知因素的评估。

三个问题决定潜变量模型是否有用。**表示：**$z$ 保留了 $x$ 的哪些信息？**推断：**$p(z\mid x)$ 能否被计算或近似？**生成：**是否定义了可以采样有效 $z$ 的先验？确定性嵌入可能只回答第一个问题，另外两个仍然没有定义。

在数字实验中，二维编码故意施加强压缩。它可以保留大体形状和类别邻域，却无法编码每个笔画细节。后续密度模型将估计这些编码的分布；它们建模的是自编码器表示，而不是直接建模原始图像。


### **确定性自编码器** {#deterministic-autoencoders}

自编码器学习编码器 $z=f_{\phi}(x)$ 和解码器 $\hat{x}=g_{\theta}(z)$，并最小化重构误差。对于固定方差的高斯观测，均方误差与负对数似然成比例：

$$
\mathcal{L}_{\mathrm{AE}}=\frac{1}{N}\sum_n\lVert x^{(n)}-g_{\theta}(f_{\phi}(x^{(n)}))\rVert_2^2.
$$

![确定性自编码器能够压缩和重构，却没有为编码定义先验。](assets/dl15-autoencoder-bottleneck.svg){fig-align="center" width="76%" fig-alt="输入依次通过编码器、二维瓶颈与解码器；最后的警告指出潜变量没有可用于采样的先验。"}

*原创教学图。*

只有当容量和正则化确实形成限制时，瓶颈才能阻止平凡的恒等映射；过完备网络仍可能记忆训练数据。重构质量也不意味着潜空间平滑或可采样：编码样本之间的任意点可能因训练从未约束这些区域而解码失败。

<details>
<summary><strong>PyTorch：训练二维确定性自编码器</strong></summary>

```python
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(64, 48), nn.ReLU(), nn.Linear(48, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 48), nn.ReLU(), nn.Linear(48, 64), nn.Sigmoid())

    def forward(self, images):
        latent = self.encoder(images)
        return self.decoder(latent), latent


def reconstruction_mse(model, images):
    model.eval()
    with torch.no_grad():
        reconstruction, _ = model(images)
    return float(F.mse_loss(reconstruction, images))


seed_everything(1520)
autoencoder = Autoencoder(latent_dim=2)
optimizer = torch.optim.AdamW(autoencoder.parameters(), lr=3e-3, weight_decay=1e-5)
loader = make_loader(train_x, seed=1520)
best_state, best_val_mse = copy.deepcopy(autoencoder.state_dict()), float("inf")
for _ in range(90):
    autoencoder.train()
    for (batch,) in loader:
        optimizer.zero_grad()
        reconstruction, _ = autoencoder(batch)
        loss = F.mse_loss(reconstruction, batch)
        loss.backward()
        optimizer.step()
    val_mse = reconstruction_mse(autoencoder, val_x)
    if val_mse < best_val_mse:
        best_val_mse, best_state = val_mse, copy.deepcopy(autoencoder.state_dict())
autoencoder.load_state_dict(best_state)

autoencoder.eval()
with torch.no_grad():
    train_z_raw = autoencoder.encoder(train_x)
    val_z_raw = autoencoder.encoder(val_x)
    test_z_raw = autoencoder.encoder(test_x)
latent_mean = train_z_raw.mean(0)
latent_std = train_z_raw.std(0).clamp_min(1e-5)
train_z = (train_z_raw - latent_mean) / latent_std
val_z = (val_z_raw - latent_mean) / latent_std
test_z = (test_z_raw - latent_mean) / latent_std

test_ae_mse = reconstruction_mse(autoencoder, test_x)
assert test_ae_mse < 0.08
assert torch.allclose(train_z.mean(0), torch.zeros(2), atol=1e-5)
print({"best validation MSE": round(best_val_mse, 4), "test MSE": round(test_ae_mse, 4),
       "standardized latent mean": train_z.mean(0).round(decimals=3).tolist(),
       "standardized latent std": train_z.std(0).round(decimals=3).tolist()})
```

</details>

标准化只在训练编码上拟合。得到的 `[B,2]` 张量支持可解释的密度实验，但二维瓶颈牺牲了重构细节。更大的潜编码会改善重构，同时让后续几何关系更难可视化。


### **去噪与稀疏自编码器** {#denoising-sparse-autoencoders}

去噪自编码器接收被破坏的 $\tilde{x}\sim q(\tilde{x}\mid x)$，并预测干净的 $x$。模型不能逐坐标复制输入，因此必须学习能够区分信号与破坏的规律。[去噪自编码器](https://www.jmlr.org/papers/v11/vincent10a.html) 将局部去噪行为与有用表示联系起来。

稀疏自编码器会惩罚潜变量活动，例如加入 $\lambda\lVert z\rVert_1$，或让平均激活接近一个较小目标。稀疏性能够产生选择性特征并限制有效容量，但过强正则化会丢失信息并造成失活单元。

![去噪与稀疏性约束自编码器保存稳定结构，而不是复制输入。](assets/dl15-denoising-sparse.svg){fig-align="center" width="76%" fig-alt="被破坏的输入被编码为稀疏表示，再向干净目标解码，并检查鲁棒性与稀疏度。"}

*依据 [Vincent 等人](https://www.jmlr.org/papers/v11/vincent10a.html) 的去噪目标绘制的原创流程图。*

<details>
<summary><strong>PyTorch：训练一个稀疏去噪自编码器并测试破坏鲁棒性</strong></summary>

```python
class SparseDenoisingAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(64, 48), nn.ReLU(), nn.Linear(48, latent_dim), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 48), nn.ReLU(), nn.Linear(48, 64), nn.Sigmoid())

    def forward(self, images):
        latent = self.encoder(images)
        return self.decoder(latent), latent


def corrupt(images, generator, mask_probability=0.18, noise_std=0.12):
    mask = torch.rand(images.shape, generator=generator) > mask_probability
    noise = noise_std * torch.randn(images.shape, generator=generator)
    return (images * mask + noise).clamp(0.0, 1.0)


seed_everything(1530)
denoising_ae = SparseDenoisingAE()
optimizer = torch.optim.AdamW(denoising_ae.parameters(), lr=2e-3, weight_decay=1e-5)
loader = make_loader(train_x, seed=1530)
noise_generator = torch.Generator().manual_seed(1530)
for _ in range(70):
    denoising_ae.train()
    for (clean_batch,) in loader:
        noisy_batch = corrupt(clean_batch, noise_generator)
        optimizer.zero_grad()
        reconstruction, latent = denoising_ae(noisy_batch)
        loss = F.mse_loss(reconstruction, clean_batch) + 8e-4 * latent.abs().mean()
        loss.backward()
        optimizer.step()

test_noise = corrupt(test_x, torch.Generator().manual_seed(1531))
denoising_ae.eval()
with torch.no_grad():
    denoised, test_sparse_z = denoising_ae(test_noise)
    noisy_mse = float(F.mse_loss(test_noise, test_x))
    denoised_mse = float(F.mse_loss(denoised, test_x))
    inactive_fraction = float((test_sparse_z < 1e-3).float().mean())

assert denoised_mse < noisy_mse
print({"corrupted-input MSE": round(noisy_mse, 4), "denoised MSE": round(denoised_mse, 4),
       "near-zero latent fraction": round(inactive_fraction, 3)})
```

</details>

破坏分布定义了模型被教会的“不变性”。掩码与高斯噪声适合本次演示，却不能代表所有真实传感器故障。去噪性能必须在与部署相关的破坏上测试，否则鲁棒性结论会被错置。


### **变分自编码器** {#variational-autoencoders}

变分自编码器（VAE）把自编码器转化为概率潜变量模型。生成模型规定 $p(z)$ 和 $p_{\theta}(x\mid z)$。由于后验

$$
p_{\theta}(z\mid x)=\frac{p_{\theta}(x\mid z)p(z)}{p_{\theta}(x)}
$$

通常不可处理，编码器 $q_{\phi}(z\mid x)$ 被用来近似它。常见选择是对角高斯分布，其 $\mu_{\phi}(x)$ 与 $\log\sigma_{\phi}^2(x)$ 由编码器输出。生成时先采样 $z\sim p(z)=\mathcal{N}(0,I)$，再从 $p_{\theta}(x\mid z)$ 采样或解码。

![VAE 把近似推断模型、概率解码器与先验连接起来。](assets/dl15-vae-graph.svg){fig-align="center" width="76%" fig-alt="观测输入进入近似推断 q，产生受先验约束的潜变量 z；解码器把 z 映射为生成观测。"}

*依据 [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114) 绘制的原创图模型解释。*

观测模型是一项建模决策。这里解码器输出归一化像素强度上固定方差高斯分布的均值。它计算透明，却会模糊多模态细节。Bernoulli 似然适用于二值像素，并不自动适用于任意连续图像。

<details>
<summary><strong>PyTorch：训练高斯解码器变分自编码器</strong></summary>

```python
class VariationalAutoencoder(nn.Module):
    def __init__(self, latent_dim=6):
        super().__init__()
        self.latent_dim = latent_dim
        self.backbone = nn.Sequential(nn.Linear(64, 56), nn.ReLU())
        self.mu = nn.Linear(56, latent_dim)
        self.logvar = nn.Linear(56, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 56), nn.ReLU(), nn.Linear(56, 64), nn.Sigmoid())

    def encode(self, images):
        hidden = self.backbone(images)
        return self.mu(hidden), self.logvar(hidden).clamp(-8.0, 6.0)

    def reparameterize(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

    def forward(self, images):
        mu, logvar = self.encode(images)
        latent = self.reparameterize(mu, logvar)
        return self.decoder(latent), mu, logvar


decoder_sigma = 0.20


def vae_terms(reconstruction, target, mu, logvar):
    reconstruction_nll = (
        0.5 * ((target - reconstruction) / decoder_sigma).pow(2)
        + math.log(decoder_sigma) + 0.5 * math.log(2 * math.pi)
    ).sum(dim=1)
    kl = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=1)
    return reconstruction_nll, kl


def train_vae(beta, seed, epochs=65, warmup=False):
    seed_everything(seed)
    model = VariationalAutoencoder()
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-5)
    loader = make_loader(train_x, seed=seed)
    for epoch in range(epochs):
        model.train()
        effective_beta = beta * min(1.0, (epoch + 1) / 18) if warmup else beta
        for (batch,) in loader:
            optimizer.zero_grad()
            reconstruction, mu, logvar = model(batch)
            reconstruction_nll, kl = vae_terms(reconstruction, batch, mu, logvar)
            loss = (reconstruction_nll + effective_beta * kl).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
    return model


def evaluate_vae(model, images):
    model.eval()
    with torch.no_grad():
        mu, logvar = model.encode(images)
        reconstruction = model.decoder(mu)
        reconstruction_nll, kl = vae_terms(reconstruction, images, mu, logvar)
    return float(reconstruction_nll.mean()), float(kl.mean()), mu, logvar


vae = train_vae(beta=1.0, seed=1540, warmup=True)
vae_reconstruction_nll, vae_kl, test_mu, test_logvar = evaluate_vae(vae, test_x)
with torch.no_grad():
    prior_samples = vae.decoder(torch.randn(40, vae.latent_dim))

assert prior_samples.shape == (40, 64)
assert math.isfinite(vae_reconstruction_nll + vae_kl)
print({"test reconstruction NLL": round(vae_reconstruction_nll, 2),
       "test KL": round(vae_kl, 2), "prior sample range":
       (round(float(prior_samples.min()), 3), round(float(prior_samples.max()), 3))})
```

</details>

有限的目标值与合法的输出范围不能证明样本质量。固定解码器方差、潜变量维度、先验和网络容量都会改变学习结果。报告 ELBO 时应同时说明这些选择，不能把“VAE loss”当作与模型无关的指标。

这里的重构 NLL 可以为负，因为方差较小时，连续概率**密度**可以大于 1；只有密度的积分必须等于 1。这并不表示概率小于零。改变 `decoder_sigma` 会改变数值密度，也会改变报告的 NLL。


### **证据下界** {#evidence-lower-bound}

引入任意近似后验 $q_{\phi}(z\mid x)$ 并应用 Jensen 不等式：

$$
\log p_{\theta}(x)
\ge
\mathbb{E}_{q_{\phi}(z\mid x)}[\log p_{\theta}(x\mid z)]
-\mathrm{KL}(q_{\phi}(z\mid x)\|p(z))
=\mathcal{L}_{\mathrm{ELBO}}.
$$

精确差距为

$$
\log p_{\theta}(x)-\mathcal{L}_{\mathrm{ELBO}}
=\mathrm{KL}(q_{\phi}(z\mid x)\|p_{\theta}(z\mid x))\ge0.
$$

因此，较差的 ELBO 可能来自较差的生成模型、不准确的推断族，或两者共同作用。重构项鼓励 $z$ 保存与 $x$ 有关的信息；先验 KL 限制信息量，并使编码区域与先验采样兼容。

![ELBO 组合期望重构与 KL 代价，并构成对数证据的下界。](assets/dl15-elbo.svg){fig-align="center" width="78%" fig-alt="期望重构减去后验到先验的 KL，得到证据下界。"}

*依据 [Kingma 与 Welling](https://arxiv.org/abs/1312.6114) 绘制的原创 ELBO 分解图。*

<details>
<summary><strong>PyTorch：逐观测分解留出集 ELBO</strong></summary>

```python
vae.eval()
with torch.no_grad():
    mu, logvar = vae.encode(test_x)
    generator = torch.Generator().manual_seed(1550)
    epsilon = torch.randn(mu.shape, generator=generator)
    latent = mu + torch.exp(0.5 * logvar) * epsilon
    reconstruction = vae.decoder(latent)
    heldout_reconstruction_nll, heldout_kl = vae_terms(reconstruction, test_x, mu, logvar)
    negative_elbo = heldout_reconstruction_nll + heldout_kl

assert torch.all(heldout_kl >= -1e-5)
assert torch.allclose(negative_elbo, heldout_reconstruction_nll + heldout_kl)
print({"mean negative ELBO": round(float(negative_elbo.mean()), 2),
       "reconstruction contribution": round(float(heldout_reconstruction_nll.mean()), 2),
       "KL contribution": round(float(heldout_kl.mean()), 2),
       "KL share of absolute objective": round(float(heldout_kl.abs().mean() /
                                                       (heldout_reconstruction_nll.abs().mean() + heldout_kl.abs().mean())), 3)})
```

</details>

ELBO 只在给定似然与变分族下构成下界。当图像缩放、解码器方差、去量化或似然族不同时，ELBO 不能直接比较。多个重要性加权样本可以收紧评估下界，但更紧的估计器无法修复较差的模型。


### **重参数化技巧** {#reparameterization-trick}

从 $q_{\phi}(z\mid x)$ 采样 $z$ 看似会阻断普通反向传播。对于对角高斯，可以写成

$$
\epsilon\sim\mathcal{N}(0,I),\qquad
z=\mu_{\phi}(x)+\sigma_{\phi}(x)\odot\epsilon.
$$

随机性被隔离在无参数的 $\epsilon$ 中，而 $z$ 对 $\mu$ 与 $\sigma$ 可微。这会产生路径导数估计器；对于连续且可重参数化的分布，其方差通常低于 score-function 估计器。

![重参数化把随机潜变量表示成参数与无参数噪声的确定性函数。](assets/dl15-reparameterization.svg){fig-align="center" width="76%" fig-alt="编码器输出 mu 与 log variance，外部高斯 epsilon 进入 z 等于 mu 加 sigma epsilon，梯度经 z 返回编码器。"}

*依据 [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114) 绘制的原创计算图。*

<details>
<summary><strong>PyTorch：验证路径梯度并比较 sample 与 rsample</strong></summary>

```python
seed_everything(1560)
demo_mu = torch.tensor([[0.3, -0.2]], requires_grad=True)
demo_logvar = torch.tensor([[0.1, -0.4]], requires_grad=True)
distribution = torch.distributions.Normal(demo_mu, torch.exp(0.5 * demo_logvar))

pathwise_z = distribution.rsample()
pathwise_loss = pathwise_z.pow(2).sum()
pathwise_loss.backward()
mu_gradient = demo_mu.grad.detach().clone()
logvar_gradient = demo_logvar.grad.detach().clone()

demo_mu_2 = torch.tensor([[0.3, -0.2]], requires_grad=True)
ordinary_sample = torch.distributions.Normal(demo_mu_2, torch.ones_like(demo_mu_2)).sample()

assert pathwise_z.requires_grad
assert not ordinary_sample.requires_grad
assert mu_gradient.abs().sum() > 0 and logvar_gradient.abs().sum() > 0
print({"rsample requires grad": pathwise_z.requires_grad,
       "sample requires grad": ordinary_sample.requires_grad,
       "mu gradient": mu_gradient.round(decimals=3).tolist(),
       "log-variance gradient": logvar_gradient.round(decimals=3).tolist()})
```

</details>

重参数化不会消除 Monte Carlo 方差，而是改变梯度估计器。离散变量通常需要松弛、straight-through 估计、边缘化或 score-function 梯度等其他工具；每种工具都有自己的偏差与方差权衡。


### **Beta-VAE 与解耦表示** {#beta-vae-disentanglement}

$\beta$-VAE 修改目标为

$$
\mathcal{L}_{\beta}=\mathbb{E}_{q(z\mid x)}[-\log p(x\mid z)]
+\beta\,\mathrm{KL}(q(z\mid x)\|p(z)).
$$

$\beta>1$ 更强地限制潜空间容量，可能鼓励因子化编码，但通常会恶化重构。[原始 $\beta$-VAE 研究](https://openreview.net/references/pdf?id=B1-vyHvOe) 在具有已知生成因素的数据上评估解耦。没有归纳假设时，无监督解耦并不可识别；仅观察潜坐标遍历图，不能证明坐标对应真实的独立原因。

实验在相同划分上比较 $\beta=0.25$、$1$ 与 $4$。线性标签探针用于检查潜均值是否保留数字类别信息；它只是诊断工具，不参与 VAE 训练，也不是一般的解耦指标。

<details>
<summary><strong>PyTorch：测量不同 beta 下的重构、rate 与探针权衡</strong></summary>

```python
low_beta_vae = train_vae(beta=0.25, seed=1570, epochs=55)
high_beta_vae = train_vae(beta=4.0, seed=1571, epochs=55)
beta_models = {0.25: low_beta_vae, 1.0: vae, 4.0: high_beta_vae}
beta_results = {}
for beta, model in beta_models.items():
    reconstruction_nll, kl, _, _ = evaluate_vae(model, test_x)
    model.eval()
    with torch.no_grad():
        train_mu, _ = model.encode(train_x)
        test_mu_beta, _ = model.encode(test_x)
    probe = LogisticRegression(max_iter=500, random_state=1570).fit(train_mu.numpy(), train_y.numpy())
    probe_accuracy = accuracy_score(test_y.numpy(), probe.predict(test_mu_beta.numpy()))
    beta_results[beta] = {"reconstruction NLL": reconstruction_nll, "KL": kl,
                          "linear probe accuracy": probe_accuracy}

for beta, row in beta_results.items():
    print({"beta": beta, **{key: round(value, 3) for key, value in row.items()}})

assert beta_results[4.0]["KL"] < beta_results[0.25]["KL"]
```

</details>

KL 经常被称为 **rate**，期望重构代价则称为 **distortion**。改变 $\beta$ 是沿 rate-distortion 前沿移动，而不是单调“改善”表示。应根据下游需求选择工作点；声称解耦时，还需要评估已知因素、干预或任务效用。


### **向量量化 VAE** {#vector-quantized-vae}

VQ-VAE 把编码器输出 $z_e(x)$ 映射到最近的已学习 codebook 向量 $e_k$：

$$
k^{*}=\arg\min_k\lVert z_e(x)-e_k\rVert_2^2,\qquad z_q(x)=e_{k^{*}}.
$$

离散索引可以再由自回归先验建模，解码器则接收量化向量。由于最近邻选择的梯度为零或不存在，straight-through 估计器把解码器梯度复制给编码器。独立的 codebook 与 commitment loss 会让嵌入靠近编码器输出，并让编码器输出停留在所选嵌入附近。

![VQ-VAE 在解码前用已学习离散 codebook 量化连续编码器输出。](assets/dl15-vqvae.svg){fig-align="center" width="76%" fig-alt="编码器输出选择最近的 codebook 向量，产生离散编码和通过 straight-through 传入解码器的向量。"}

*依据 [Neural Discrete Representation Learning](https://arxiv.org/abs/1711.00937) 绘制的原创机制图。*

<details>
<summary><strong>PyTorch：实现 codebook 查找、straight-through 梯度与利用率诊断</strong></summary>

```python
class VQVAE(nn.Module):
    def __init__(self, codebook_size=24, embedding_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, embedding_dim))
        self.codebook = nn.Embedding(codebook_size, embedding_dim)
        nn.init.uniform_(self.codebook.weight, -0.35, 0.35)
        self.decoder = nn.Sequential(nn.Linear(embedding_dim, 64), nn.ReLU(), nn.Linear(64, 64), nn.Sigmoid())

    def forward(self, images):
        encoded = self.encoder(images)
        distances = (encoded.pow(2).sum(1, keepdim=True)
                     - 2 * encoded @ self.codebook.weight.T
                     + self.codebook.weight.pow(2).sum(1).unsqueeze(0))
        indices = distances.argmin(dim=1)
        quantized = self.codebook(indices)
        straight_through = encoded + (quantized - encoded).detach()
        reconstruction = self.decoder(straight_through)
        return reconstruction, encoded, quantized, indices


seed_everything(1580)
vqvae = VQVAE()
loader = make_loader(train_x, seed=1580)
# First learn a continuous bottleneck so that codebook initialization sees the data geometry.
pretrain_optimizer = torch.optim.AdamW(
    list(vqvae.encoder.parameters()) + list(vqvae.decoder.parameters()), lr=2e-3
)
for _ in range(35):
    for (batch,) in loader:
        pretrain_optimizer.zero_grad()
        encoded = vqvae.encoder(batch)
        reconstruction = vqvae.decoder(encoded)
        F.mse_loss(reconstruction, batch).backward()
        pretrain_optimizer.step()

with torch.no_grad():
    initial_codes = vqvae.encoder(train_x).numpy()
centers = KMeans(
    n_clusters=vqvae.codebook.num_embeddings, n_init=10, random_state=1580
).fit(initial_codes).cluster_centers_
with torch.no_grad():
    vqvae.codebook.weight.copy_(torch.tensor(centers, dtype=torch.float32))

optimizer = torch.optim.AdamW(vqvae.parameters(), lr=1e-3, weight_decay=1e-5)
for _ in range(65):
    vqvae.train()
    for (batch,) in loader:
        optimizer.zero_grad()
        reconstruction, encoded, quantized, _ = vqvae(batch)
        reconstruction_loss = F.mse_loss(reconstruction, batch)
        codebook_loss = F.mse_loss(quantized, encoded.detach())
        commitment_loss = F.mse_loss(encoded, quantized.detach())
        loss = reconstruction_loss + codebook_loss + 0.25 * commitment_loss
        loss.backward()
        optimizer.step()

vqvae.eval()
with torch.no_grad():
    vq_reconstruction, _, _, vq_indices = vqvae(test_x)
    vq_mse = float(F.mse_loss(vq_reconstruction, test_x))
    usage = torch.bincount(vq_indices, minlength=vqvae.codebook.num_embeddings).float()
    usage_probability = usage / usage.sum()
    codebook_perplexity = float(torch.exp(-(usage_probability[usage > 0] *
                                             usage_probability[usage > 0].log()).sum()))

assert vq_indices.dtype == torch.long
print({"test reconstruction MSE": round(vq_mse, 4),
       "used codes": int((usage > 0).sum()),
       "codebook size": vqvae.codebook.num_embeddings,
       "codebook perplexity": round(codebook_perplexity, 2)})
assert int((usage > 0).sum()) >= 4
```

</details>

当只有少数条目被使用时，就发生 codebook collapse。应监控使用直方图与 perplexity；指数移动平均更新、code reset、更大 batch 或 commitment 调整都可能有帮助。只有 VQ-VAE 解码器还不是完整生成器，必须另行学习 code 索引上的先验。


### **后验坍塌** {#posterior-collapse}

当 $q_{\phi}(z\mid x)\approx p(z)$ 且解码器忽略 $z$ 时，就发生后验坍塌。此时每个样本的 KL 接近零，潜均值在数据间变化很小，替换 $z$ 也几乎不影响输出。强大的自回归解码器尤其容易出现这一问题，因为它可以只依靠先前输出建模 $x$，不使用潜通道。

![信息性后验随输入变化，坍塌后验则匹配先验并失去有效潜通道。](assets/dl15-posterior-collapse.svg){fig-align="center" width="76%" fig-alt="面板比较信息性与坍塌后验，并列出逐维 KL、活跃单元和解码器敏感度诊断。"}

*参考 [Lagging Inference Networks and Posterior Collapse](https://arxiv.org/abs/1901.05534) 绘制的原创诊断图。*

KL annealing、free bits、削弱解码器、从 $z$ 加入跳跃连接、更富表达力的后验，以及额外推断更新都可能有帮助，但每种方法都会改变优化或建模假设。非零总 KL 仍可能掩盖局部坍塌，因此需要逐维检查 KL 与方差。

<details>
<summary><strong>PyTorch：诊断活跃潜变量单元与解码器敏感度</strong></summary>

```python
def collapse_diagnostics(model, images):
    model.eval()
    with torch.no_grad():
        mu, logvar = model.encode(images)
        kl_per_dimension = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).mean(0)
        active_units = int((mu.var(0) > 1e-2).sum())
        original = model.decoder(mu)
        shuffled = model.decoder(mu[torch.randperm(len(mu), generator=torch.Generator().manual_seed(1590))])
        decoder_sensitivity = float((original - shuffled).pow(2).mean())
    return kl_per_dimension, active_units, decoder_sensitivity


standard_kl_dims, standard_active, standard_sensitivity = collapse_diagnostics(vae, test_x)
high_kl_dims, high_active, high_sensitivity = collapse_diagnostics(high_beta_vae, test_x)
print({"model": "beta=1", "active units": standard_active,
       "mean KL per dim": round(float(standard_kl_dims.mean()), 3),
       "decoder sensitivity": round(standard_sensitivity, 4)})
print({"model": "beta=4", "active units": high_active,
       "mean KL per dim": round(float(high_kl_dims.mean()), 3),
       "decoder sensitivity": round(high_sensitivity, 4)})

assert standard_active <= vae.latent_dim and high_active <= high_beta_vae.latent_dim
assert standard_sensitivity >= 0 and high_sensitivity >= 0
```

</details>

这里用较高 $\beta$ 演示容量压力，并不声称发生了完全坍塌。严格研究应在训练过程中持续追踪诊断、控制解码器容量，并测试潜变量干预是否改变生成输出。当真实任务只需要很少潜信息时，较低 KL 也可能是合理结果。


### **归一化流** {#normalizing-flows}

令 $z=f_{\theta}(x)$ 可逆，且 $p_Z(z)$ 可处理，则变量替换公式给出

$$
\log p_X(x)=\log p_Z(f_{\theta}(x))+
\log\left|\det\frac{\partial f_{\theta}(x)}{\partial x}\right|.
$$

与 VAE 不同，流在连续密度假设下提供精确潜变量推断和精确似然。可逆性会保持维度，并限制可用架构。一般 Jacobian 行列式需要 $O(D^3)$，因此 flow layer 会使用三角、自回归或其他结构化 Jacobian。

RealNVP affine coupling 保持一个分区不变，并用它缩放和平移另一个分区。其 Jacobian 为三角矩阵，因此 log determinant 是 scale 输出之和，逆变换也有解析形式。

![RealNVP affine coupling 具有三角 Jacobian、可处理 log determinant 与精确逆变换。](assets/dl15-realnvp.svg){fig-align="center" width="78%" fig-alt="三个面板展示 affine coupling 方程、变量替换密度与精确逆变换。"}

*依据 [Density Estimation using RealNVP](https://arxiv.org/abs/1605.08803) 绘制的原创推导图。*

<details>
<summary><strong>PyTorch：在共享二维编码上拟合可逆 RealNVP 密度</strong></summary>

```python
class AffineCoupling(nn.Module):
    def __init__(self, mask):
        super().__init__()
        self.register_buffer("mask", torch.tensor(mask, dtype=torch.float32))
        self.network = nn.Sequential(nn.Linear(2, 48), nn.Tanh(), nn.Linear(48, 48),
                                     nn.Tanh(), nn.Linear(48, 4))

    def forward(self, inputs):
        fixed = inputs * self.mask
        scale, shift = self.network(fixed).chunk(2, dim=1)
        scale = 1.4 * torch.tanh(scale) * (1 - self.mask)
        shift = shift * (1 - self.mask)
        outputs = fixed + (1 - self.mask) * (inputs * torch.exp(scale) + shift)
        return outputs, scale.sum(dim=1)

    def inverse(self, outputs):
        fixed = outputs * self.mask
        scale, shift = self.network(fixed).chunk(2, dim=1)
        scale = 1.4 * torch.tanh(scale) * (1 - self.mask)
        shift = shift * (1 - self.mask)
        return fixed + (1 - self.mask) * (outputs - shift) * torch.exp(-scale)


class RealNVP(nn.Module):
    def __init__(self, layers=6):
        super().__init__()
        self.layers = nn.ModuleList([
            AffineCoupling([1, 0] if index % 2 == 0 else [0, 1]) for index in range(layers)
        ])

    def transform(self, inputs):
        latent, logdet = inputs, torch.zeros(len(inputs))
        for layer in self.layers:
            latent, contribution = layer(latent)
            logdet += contribution
        return latent, logdet

    def log_prob(self, inputs):
        base, logdet = self.transform(inputs)
        base_log_prob = -0.5 * (base.pow(2) + math.log(2 * math.pi)).sum(dim=1)
        return base_log_prob + logdet

    def sample(self, count, seed=1600):
        generator = torch.Generator().manual_seed(seed)
        outputs = torch.randn((count, 2), generator=generator)
        for layer in reversed(self.layers):
            outputs = layer.inverse(outputs)
        return outputs


seed_everything(1600)
flow = RealNVP()
optimizer = torch.optim.AdamW(flow.parameters(), lr=2e-3, weight_decay=1e-5)
generator = torch.Generator().manual_seed(1600)
for _ in range(550):
    indices = torch.randint(len(train_z), (256,), generator=generator)
    optimizer.zero_grad()
    loss = -flow.log_prob(train_z[indices]).mean()
    loss.backward()
    optimizer.step()

flow.eval()
with torch.no_grad():
    flow_test_nll = float(-flow.log_prob(test_z).mean())
    flow_samples = flow.sample(300)
    transformed, _ = flow.transform(test_z[:32])
    reconstructed = transformed
    for layer in reversed(flow.layers):
        reconstructed = layer.inverse(reconstructed)
    inversion_error = float((reconstructed - test_z[:32]).abs().max())

assert inversion_error < 1e-4
assert torch.isfinite(flow_samples).all()
print({"test latent NLL": round(flow_test_nll, 3), "max inversion error": inversion_error,
       "sample mean": flow_samples.mean(0).round(decimals=3).tolist(),
       "sample std": flow_samples.std(0).round(decimals=3).tolist()})
```

</details>

报告的似然属于经过训练集标准化的二维自编码器编码，而不是原始图像。由于确定性编码器不可逆，这个 flow 无法分配图像空间似然。它是对 flow 机制的受控演示，也提醒我们似然单位取决于被建模的空间。


### **能量模型** {#energy-based-models}

能量模型定义

$$
p_{\theta}(x)=\frac{\exp[-E_{\theta}(x)]}{Z_{\theta}},\qquad
Z_{\theta}=\int \exp[-E_{\theta}(x)]\,dx.
$$

$E_{\theta}(x)$ 可以非常灵活，但配分函数 $Z_{\theta}$ 会耦合所有可能配置。最大似然梯度包含降低数据能量的正相，以及提高模型样本能量的负相。获得负相通常需要 MCMC、近似或替代目标。

可执行示例使用噪声对比估计（NCE）：分类器区分数据编码与已知噪声密度 $q(z)$ 的样本。在最优点，其 logit 估计 log density ratio，从而能够在相差常数的意义下恢复未归一化数据 log density 与能量。这样可以明确展示训练信号，而不声称实现了精确最大似然。

![能量定义地形，其负梯度是 score，Langevin dynamics 把 score 漂移与噪声结合。](assets/dl15-energy-score.svg){fig-align="center" width="78%" fig-alt="能量地形包含低能量数据盆地；score 场指向更高密度，Langevin dynamics 再加入高斯噪声。"}

*依据 [Energy-Based Learning 教程](https://yann.lecun.org/exdb/publis/pdf/lecun-06.pdf) 绘制的原创综合图。*

<details>
<summary><strong>PyTorch：用 NCE 学习潜空间能量并通过 Langevin dynamics 采样</strong></summary>

```python
class DensityRatio(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(2, 64), nn.SiLU(), nn.Linear(64, 64),
                                     nn.SiLU(), nn.Linear(64, 1))

    def forward(self, points):
        return self.network(points).squeeze(1)


noise_scale = 2.2


def noise_log_prob(points):
    return -0.5 * ((points / noise_scale).pow(2) + math.log(2 * math.pi * noise_scale**2)).sum(1)


seed_everything(1610)
ratio_model = DensityRatio()
optimizer = torch.optim.AdamW(ratio_model.parameters(), lr=2e-3, weight_decay=1e-4)
generator = torch.Generator().manual_seed(1610)
for _ in range(650):
    indices = torch.randint(len(train_z), (192,), generator=generator)
    data_batch = train_z[indices]
    noise_batch = noise_scale * torch.randn((192, 2), generator=generator)
    points = torch.cat([data_batch, noise_batch])
    targets = torch.cat([torch.ones(192), torch.zeros(192)])
    optimizer.zero_grad()
    loss = F.binary_cross_entropy_with_logits(ratio_model(points), targets)
    loss.backward()
    optimizer.step()


def unnormalized_log_density(points):
    return ratio_model(points) + noise_log_prob(points)


def langevin_energy_samples(count=300, steps=120, step_size=0.025, seed=1611):
    generator = torch.Generator().manual_seed(seed)
    points = noise_scale * torch.randn((count, 2), generator=generator)
    for _ in range(steps):
        points.requires_grad_(True)
        log_density = unnormalized_log_density(points).sum()
        score = torch.autograd.grad(log_density, points)[0]
        noise = torch.randn(points.shape, generator=generator)
        points = (points + step_size * score + math.sqrt(2 * step_size) * noise).detach().clamp(-6, 6)
    return points


ratio_model.eval()
energy_samples = langevin_energy_samples()
with torch.no_grad():
    data_energy = float((-unnormalized_log_density(test_z)).mean())
    broad_noise = noise_scale * torch.randn((len(test_z), 2), generator=torch.Generator().manual_seed(1612))
    noise_energy = float((-unnormalized_log_density(broad_noise)).mean())

assert data_energy < noise_energy
assert energy_samples.shape == (300, 2)
print({"mean data energy": round(data_energy, 3), "mean broad-noise energy": round(noise_energy, 3),
       "Langevin sample mean": energy_samples.mean(0).round(decimals=3).tolist()})
```

</details>

短程 Langevin 样本可能受到初始化、步长、混合速度与多模态障碍的偏置。能量值只能确定到一个加法常数。诊断应包含不同初始化的多条链、自相关、有效样本量和留出任务，而不是只展示一张好看的散点图。


### **Score Matching** {#score-matching}

连续密度的 score 为

$$
s_{\theta}(x)=\nabla_x\log p_{\theta}(x)=-\nabla_x E_{\theta}(x).
$$

未知配分函数会在梯度中消失。经典 [score matching](https://jmlr.org/papers/v6/hyvarinen05a.html) 无需计算 $Z_{\theta}$，即可拟合模型 score 与数据 score。Denoising score matching 先用高斯噪声破坏 $x$，再预测带噪条件分布的 score。对于 $\tilde{x}=x+\epsilon$、$\epsilon\sim\mathcal{N}(0,\sigma^2I)$，目标是

$$
\nabla_{\tilde{x}}\log q(\tilde{x}\mid x)
=-\frac{\tilde{x}-x}{\sigma^2}=-\frac{\epsilon}{\sigma^2}.
$$

学到的向量场会把带噪点推向高斯平滑数据分布中密度更高的区域。单一噪声级别在 $\sigma$ 较大时会丢失细节，在 $\sigma$ 较小时又难以处理薄流形。第 16 章将把这一思想扩展到多个噪声级别和反向扩散。

<details>
<summary><strong>PyTorch：在相同潜分布上学习 denoising score 场</strong></summary>

```python
class ScoreNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(2, 64), nn.Tanh(), nn.Linear(64, 64),
                                     nn.Tanh(), nn.Linear(64, 2))

    def forward(self, points):
        return self.network(points)


score_sigma = 0.35
seed_everything(1620)
score_model = ScoreNetwork()
optimizer = torch.optim.AdamW(score_model.parameters(), lr=2e-3, weight_decay=1e-5)
generator = torch.Generator().manual_seed(1620)
for _ in range(700):
    indices = torch.randint(len(train_z), (256,), generator=generator)
    clean = train_z[indices]
    epsilon = score_sigma * torch.randn(clean.shape, generator=generator)
    noisy = clean + epsilon
    target_score = -epsilon / score_sigma**2
    optimizer.zero_grad()
    score_loss = F.mse_loss(score_model(noisy), target_score)
    score_loss.backward()
    optimizer.step()

with torch.no_grad():
    validation_epsilon = score_sigma * torch.randn(val_z.shape, generator=torch.Generator().manual_seed(1621))
    validation_score_loss = float(F.mse_loss(
        score_model(val_z + validation_epsilon), -validation_epsilon / score_sigma**2
    ))


def score_langevin(count=300, steps=140, step_size=0.012, seed=1622):
    generator = torch.Generator().manual_seed(seed)
    points = 2.2 * torch.randn((count, 2), generator=generator)
    initial = points.clone()
    for _ in range(steps):
        with torch.no_grad():
            drift = score_model(points)
        noise = torch.randn(points.shape, generator=generator)
        points = (points + step_size * drift + math.sqrt(2 * step_size) * noise).clamp(-6, 6)
    return initial, points


score_initial, score_samples = score_langevin()
initial_distance = torch.cdist(score_initial, train_z).min(1).values.mean()
final_distance = torch.cdist(score_samples, train_z).min(1).values.mean()
assert torch.isfinite(score_samples).all()
print({"validation denoising score MSE": round(validation_score_loss, 3),
       "initial nearest-data distance": round(float(initial_distance), 3),
       "final nearest-data distance": round(float(final_distance), 3)})
```

</details>

最近数据距离需要谨慎解释：靠近训练点可能表示学到了真实结构，也可能表示记忆。Score 估计误差、采样器离散误差与混合误差彼此独立。在单个 $\sigma$ 上获得较低去噪损失，并不表示跨尺度密度已经校准。


### **潜变量、流与能量模型对比** {#latent-flow-energy-models-compared}

这些模型族最重要的差异不是笼统的“生成质量”，而是各自让哪种计算变得可处理。

| 模型族 | 密度访问 | 推断 | 采样 | 主要结构代价 |
|---|---|---|---|---|
| 确定性 AE | 没有归一化密度 | 一次编码器前向传播 | 没有潜先验时未定义 | 重构目标可能让潜空间出现空洞 |
| VAE | ELBO / 近似边缘似然 | 摊销 $q_{\phi}(z\mid x)$ | 先验采样加解码器 | 变分差距与 rate-distortion 权衡 |
| VQ-VAE | 离散编码；另行学习先验 | 最近 codebook 查找 | 需要 code prior | codebook collapse 与 straight-through 偏差 |
| 归一化流 | 精确连续似然 | 精确正向/逆向映射 | 精确逆变换 | 可逆性、相同维度与 Jacobian 设计 |
| EBM | 未归一化密度 | 优化或条件采样 | 通常需要迭代 MCMC | 配分函数与混合问题 |
| Score model | 密度梯度，默认没有逐点密度 | 向量场求值 | 迭代随机动力学 | 噪声尺度覆盖与数值求解误差 |

在本章实验中，自编码器编码是共享测量空间。Flow 只在该空间具有精确似然；NCE 能量与 denoising score 避免显式归一化，却依赖近似采样；VAE 通过潜变量不确定性直接建模图像，但优化的是下界。直接比较它们的原始 loss 没有意义，因为目标与单位不同。

模型选择应从所需操作开始：需要摊销推断与结构化随机瓶颈时使用 VAE；需要离散可复用单元时考虑 VQ-VAE；精确连续似然与可逆性值得架构限制时使用 flow；需要灵活兼容函数或条件推断时考虑 EBM；当 log density 梯度和迭代细化是核心时使用 score model。


### **章节对比与总结** {#chapter-comparison-summary}

本章用三种更广泛的建模策略替代单一有序分解。潜变量模型通过隐藏结构压缩并解释观测；变分推断把不可处理后验转化为可优化 ELBO；flow 通过可逆性保存精确密度；能量与 score 模型则用灵活地形和梯度采样交换归一化似然。

共享 UCI 实验揭示了操作差异。确定性瓶颈能够重构数字，但采样前还需要额外密度；去噪与稀疏性加入有用不变性和容量限制；VAE 实验分离重构与 KL rate、验证路径梯度，并诊断 $\beta$ 的作用；VQ-VAE 让 codebook utilization 可测量；RealNVP 对可逆性做了单元测试；NCE 相对已知噪声学习能量，denoising score matching 则在同一潜分布上学习局部向量场。

实际审查需要询问：假设了哪些随机变量和观测似然？潜变量推断是精确、近似还是未定义？似然在哪个空间中测量？采样需要先验、逆映射、自回归 prior 还是 MCMC？哪项近似带来偏差：变分族、straight-through 梯度、有限 flow 架构、对比目标，还是离散化动力学？最后，重构、留出密度、覆盖度、记忆与下游效用是否一致？

第 16 章将从这些基础进入对抗式与迭代式高保真生成。GAN 通过判别器学习，diffusion model 跨噪声尺度学习去噪，flow matching 学习连续输运。本章的 score-matching 部分提供概念桥梁，而不重复完整扩散推导。
